<h1><span style="color:red">Generate Factor Contributions</span></h1>

### This notebook reads numeric and categorical variables from the survey dataset and computes factor contributions for all levels of the variables in the survey dataset.

## 1. Retrieve survey parameters from the URL

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


In [ ]:
# common imports
from __future__ import print_function
import re
import math
from itertools import product as _iproduct
import base64

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, Javascript

pd.set_option('display.max_colwidth', 0)

def printmd(string):
    display(Markdown(string))

# local imports
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint


In [ ]:
# Computation helpers

def _clean(col):
    """Strip SuAVE qualifiers and __binned__ prefix."""
    return re.sub(r'__binned__|#\w+', '', col).strip()

def _fmt_bin(interval):
    """Format a pandas Interval with adaptive decimal precision."""
    if not hasattr(interval, 'left'):
        return str(interval)
    left, right = interval.left, interval.right
    w = right - left
    prec = 5 if w < 0.001 else 4 if w < 0.01 else 3 if w < 0.1 else 2 if w < 1 else 1 if w < 10 else 0
    return f'[{left:.{prec}f}, {right:.{prec}f})'

def _sort_key(x):
    """Numeric sort key for bin-label strings."""
    try:
        m = re.search(r'[-+]?[0-9]*\.?[0-9]+', str(x))
        return float(m.group()) if m else float('inf')
    except Exception:
        return float('inf')

def compute_contributions(df, target_col, target_val, expl_cols):
    """
    Enumerate every combination of values across expl_cols and compute:
      accuracy  = P(target_col == target_val | combo)
      contrib_k = accuracy - accuracy when variable k is removed from the condition
    Returns (result_df, accuracy_column_name).
    """
    acc_col = f'Accuracy \u2192 {target_val}'
    rows = []
    combos = list(_iproduct(*[df[v].dropna().unique() for v in expl_cols]))
    for combo in combos:
        combo_dict = dict(zip(expl_cols, combo))
        mask = (df[list(combo_dict)] == pd.Series(combo_dict)).all(axis=1)
        subset = df[mask]
        if len(subset) == 0:
            continue
        acc = (subset[target_col] == target_val).mean()
        rule = ' AND '.join(f'{_clean(k)}={v}' for k, v in combo_dict.items())
        row = {'Rule': rule, 'n': int(len(subset)), acc_col: round(acc, 2)}
        for var in expl_cols:
            reduced = {k: v for k, v in combo_dict.items() if k != var}
            if reduced:
                red_mask = (df[list(reduced)] == pd.Series(reduced)).all(axis=1)
            else:
                red_mask = pd.Series(True, index=df.index)
            red_sub = df[red_mask]
            acc_r = (red_sub[target_col] == target_val).mean() if len(red_sub) > 0 else None
            row[f'Contrib of {_clean(var)}'] = (
                round(acc - acc_r, 2) if acc_r is not None else None
            )
        rows.append(row)
    return pd.DataFrame(rows), acc_col


## 2. Read the survey file

In [5]:
# read the csv file
df = panellibs.extract_data(absolutePath + csv_file)

# create a list of variable names
variables_df = pd.DataFrame({'varname':df.columns})
printmd("<b><span style='color:red'>All variables in the survey file:</span></b>")
print(variables_df.varname.values)

## 3. Prepare Variables

In [ ]:
# Drop SuAVE metadata columns that are not useful for analysis
original_df = df.copy()
_drop = [c for c in df.columns
         if any(t in c for t in ('#img', '#name', '#long', '#hidden', '#hiddenmore'))]
df.drop(columns=_drop, errors='ignore', inplace=True)

numeric_cols     = [c for c in df.columns if '#number' in c]
categorical_cols = df.select_dtypes(include='object').columns.tolist()
all_vars         = categorical_cols + numeric_cols

print(f'{len(all_vars)} variables available: '
      f'{len(categorical_cols)} categorical, {len(numeric_cols)} numeric')
for v in all_vars:
    print(f'  {v}')


## 4. Bin Numeric Variables (Optional)

Numeric variables must be converted to categories before use. Run the cell below to see histograms, then edit `bin_specs` and run the next cell to apply the bins.

Leave `bin_specs` empty to use 5 equal-width bins for every numeric column.

In [ ]:
# Show histograms for all numeric columns
if numeric_cols:
    ncols_plot = min(3, len(numeric_cols))
    nrows_plot = math.ceil(len(numeric_cols) / ncols_plot)
    fig, axes = plt.subplots(nrows_plot, ncols_plot,
                             figsize=(5 * ncols_plot, 2.5 * nrows_plot))
    axes = np.array(axes).flatten()
    for ax, col in zip(axes, numeric_cols):
        df[col].dropna().hist(bins=30, ax=ax, edgecolor='black')
        ax.set_title(_clean(col), fontsize=9)
        ax.tick_params(labelsize=7)
    for ax in axes[len(numeric_cols):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print('No numeric columns in this survey.')


In [ ]:
# Edit bin_specs before running this cell.
# Keys are exact column names (with #number suffix).
# Value = int  → that many equal-width bins.
# Value = list → explicit bin edges (e.g. [0, 25, 75, 100]).
bin_specs = {
    # 'ColumnName#number': 5,
    # 'ColumnName#number': [0, 10, 50, 100],
}

rebinned = {}  # maps original col name → new __binned__ col name

for col in numeric_cols:
    spec = bin_specs.get(col, 5)
    raw  = df[col].dropna()
    edges = np.linspace(raw.min(), raw.max(), spec + 1) if isinstance(spec, int) \
            else sorted(set(spec))
    binned  = pd.cut(raw, bins=edges, include_lowest=True)
    new_col = f'__binned__{_clean(col)}'
    df[new_col] = pd.Series(binned.map(_fmt_bin), index=raw.index)
    rebinned[col] = new_col
    n_bins_used = len(edges) - 1
    print(f'  {col}  →  {n_bins_used} bins  →  {new_col}')

if not rebinned:
    print('No numeric columns to bin.')

# Build the combined variable list used by all subsequent selectors.
# Numeric originals are replaced by their binned counterparts.
_analysis_vars = [rebinned.get(c, c) for c in all_vars
                  if rebinned.get(c, c) in df.columns]


## 5. Select Target Variable (A)

The variable whose value you want to explain.

In [ ]:
_var_opts = [
    (_clean(v) + (' (binned)' if '__binned__' in v else ''), v)
    for v in _analysis_vars
]
target_selector = widgets.Dropdown(
    options=_var_opts,
    description='Target (A):',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='460px'),
)
display(target_selector)


In [ ]:
# Confirm target and select the specific value to explain
target_col = target_selector.value
vc = df[target_col].value_counts(sort=False)
sorted_vals = sorted(vc.index.tolist(), key=_sort_key)

target_value_selector = widgets.Dropdown(
    options=[(f'{v}  (n={vc[v]})', v) for v in sorted_vals],
    description=f'{_clean(target_col)}:',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='480px'),
)
display(HTML(f'<b>Target variable:</b> {target_col}  &mdash;  '
            f'{len(sorted_vals)} distinct values'))
display(target_value_selector)


## 6. Select Explanatory Variables (B, C, D...)

Select up to 4 variables. Hold **Ctrl / Cmd** to pick multiple.

In [ ]:
_expl_opts = [
    (_clean(v) + (' (binned)' if '__binned__' in v else ''), v)
    for v in _analysis_vars
    if v != target_col
]
expl_selector = widgets.SelectMultiple(
    options=_expl_opts,
    description='Explanatory:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='460px', height='180px'),
    rows=8,
)
display(expl_selector)


In [ ]:
# Confirm and validate (run after making your selection above)
expl_cols = list(expl_selector.value)[:4]
if not expl_cols:
    raise ValueError('Select at least one explanatory variable above.')
target_val = target_value_selector.value
print(f'Target   : {target_col} = "{target_val}"')
print(f'Explain  : {[_clean(c) for c in expl_cols]}')


## 7. Compute Factor Contributions

Adjust the filters below, then run the computation cell.

In [ ]:
# Filter parameters — edit before running the next cell
MIN_ROWS         = 5    # drop rules with fewer items than this
MIN_ACCURACY     = 0.0  # 0.0 – 1.0
MIN_CONTRIBUTION = 0.0  # minimum |contribution| to include (0.0 – 1.0)
SORT_BY          = 'n'  # 'n', acc_col name, or 'Contrib of X'
SORT_ASCENDING   = False


In [ ]:
result_df, acc_col = compute_contributions(df, target_col, target_val, expl_cols)

contrib_cols = [c for c in result_df.columns if c.startswith('Contrib of')]
filtered = result_df[result_df['n'] >= MIN_ROWS].copy()
filtered = filtered[filtered[acc_col] >= MIN_ACCURACY]
if MIN_CONTRIBUTION > 0:
    _cmask = (filtered[contrib_cols].abs() >= MIN_CONTRIBUTION).any(axis=1)
    filtered = filtered[_cmask]

_sort_col = SORT_BY if SORT_BY in filtered.columns else acc_col
filtered  = filtered.sort_values(_sort_col, ascending=SORT_ASCENDING)\
                     .reset_index(drop=True)

print(f'{len(result_df)} rules computed, {len(filtered)} shown after filters.')


## 8. View Results

Rows highlighted in green have accuracy > 0.8. Individual cells highlighted in yellow have |contribution| > 0.2.

In [ ]:
def _highlight(row):
    style = [''] * len(row)
    if row[acc_col] > 0.8:
        style = ['background-color: #c8f7c5'] * len(row)
    for i, col in enumerate(row.index):
        if col.startswith('Contrib of'):
            val = row[col]
            if val is not None and not (isinstance(val, float) and math.isnan(val)):
                if abs(val) > 0.2:
                    style[i] = 'background-color: #f5f5aa'
    return style

if filtered.empty:
    display(HTML('<p style="color:#e07000">No rules match the current filters.</p>'))
else:
    display(HTML(f'<p><b>{len(filtered)}</b> rules</p>'))
    display(
        filtered.style
            .apply(_highlight, axis=1)
            .format(precision=2, na_rep='—')
            .set_properties(**{'text-align': 'left'})
    )


## 9. Download Results

In [ ]:
_csv = filtered.to_csv(index=False)
_b64 = base64.b64encode(_csv.encode()).decode()
display(Javascript(f"""
var a = document.createElement('a');
a.download = 'factor_contributions.csv';
a.href = 'data:text/csv;base64,{_b64}';
document.body.appendChild(a);
a.click();
document.body.removeChild(a);
"""))
print('Download triggered.')


## 10. Save Results to SuAVE (Optional)

Append the contribution table as new columns to the survey CSV and upload.

In [ ]:
# Merge Accuracy and Contribution columns back onto the original df
# (matched on the Rule string — skip this section if you only want the download)
print('Use the Download Results cell above to get a CSV of the factor contributions table.')
print('To append new columns back to the survey, merge filtered with original_df '
      'on a shared key column and use suaveint.save_csv_file + suaveint.create_survey.')
